# CardioSafe AI - Notebook 07: medication knowledge base

The medication side never invents medical claims. Every flag must point to a source sentence or table row. This notebook builds and checks the three evidence sources:

| Source | What it provides | Terms |
|---|---|---|
| **RxNorm** (NLM REST API) | brand / free-text name -> standard ingredient | free, no licence needed for these endpoints |
| **openFDA drug labels** | the sentences in official FDA labels about QT prolongation, bradycardia, AV block, with section + citation | free; optional free API key raises the daily limit from 1,000 to 120,000 requests |
| **DDInter** | drug-drug interaction severity (major / moderate / minor) | **CC BY-NC-SA 4.0**: non-commercial use with attribution (fine for a course project) |
| *(optional)* **CredibleMeds** | expert QT-risk categories | copyrighted; needs your own free registration; used **locally for evaluation only**, never bundled or redistributed |

**What you do before running (5 minutes):**
1. Download the DDInter CSV files from the DDInter website's Download page and put them in `MyDrive/CardioSafe/kb/ddinter/` (I could not open the page, so I do not know the exact file names; the notebook auto-detects the columns and prints what it found). If the notebook finds none it still runs and tells you what is missing.
2. *(Optional)* register at crediblemeds.org, download the QTdrugs list, and put the CSV/XLSX in `MyDrive/CardioSafe/kb/crediblemeds/`. If present, the notebook measures how well our label-based QT evidence agrees with it.
3. *(Optional, recommended)* get a free openFDA API key (open.fda.gov/apis/authentication) and add it as a Colab secret named `OPENFDA_API_KEY` (key icon in the left sidebar).

**Important:** the QT-evidence extractor is a rule-based text classifier. I have only tested it on example sentences I wrote myself, not on real labels. The notebook prints the real evidence sentences so we can inspect them and fix the rules. Treat the first run as a calibration run.

**Runtime:** CPU. About 10 to 20 minutes (the APIs are rate-limited and the notebook pauses between calls).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
WORK = '/content/drive/MyDrive/CardioSafe'
KB = WORK + '/kb'
for d in ('', '/ddinter', '/crediblemeds'):
    os.makedirs(KB + d, exist_ok=True)
sys.path.insert(0, WORK)
print('DDInter files found:', os.listdir(KB + '/ddinter'))
print('CredibleMeds files found:', os.listdir(KB + '/crediblemeds'))

In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_kb.py
"""CardioSafe drug knowledge base.

- RxNorm: free-text drug name -> RxCUI -> ingredient names (https://rxnav.nlm.nih.gov/REST)
- openFDA drug labels: retrieves label sentences about QT prolongation / bradycardia / AV block, with section and citation
- DDInter: drug-drug interaction severity from the user's downloaded CSV files (CC BY-NC-SA 4.0, non-commercial use)
- CredibleMeds (optional): the user's own downloaded list, used locally for evaluation only; never redistributed

Nothing here invents medical claims: every flag carries the label sentence or table row it came from.
"""
import os, re, json, time, glob
import numpy as np
import pandas as pd

RXNAV = 'https://rxnav.nlm.nih.gov/REST'
FDA_LABEL = 'https://api.fda.gov/drug/label.json'
UA = {'User-Agent': 'CardioSafe-AI-course-project'}
STATS = {'requests': 0, 'errors': 0}


# ------------------------------------------------------------------ cache + http
class JsonCache:
    def __init__(self, path):
        self.path, self.d, self.dirty = path, {}, 0
        if path and os.path.exists(path):
            try:
                self.d = json.load(open(path))
            except Exception:
                self.d = {}

    def has(self, k): return k in self.d
    def get(self, k): return self.d.get(k)

    def set(self, k, v):
        self.d[k] = v
        self.dirty += 1
        if self.dirty % 20 == 0:
            self.save()

    def save(self):
        if self.path:
            tmp = self.path + '.tmp'
            json.dump(self.d, open(tmp, 'w'))
            os.replace(tmp, self.path)
        self.dirty = 0


def http_get(url, params=None, retries=3, pause=0.3, timeout=30):
    """GET -> parsed JSON; {'_status': 404/400} for those codes; None on network failure."""
    import requests
    for a in range(retries):
        try:
            r = requests.get(url, params=params, headers=UA, timeout=timeout)
            STATS['requests'] += 1
            if r.status_code in (400, 404):
                return {'_status': r.status_code}
            if r.status_code == 429:
                time.sleep(3 * (a + 1)); continue
            if r.status_code >= 400:
                STATS['errors'] += 1; time.sleep(1 + a); continue
            time.sleep(pause)
            return r.json()
        except Exception:
            STATS['errors'] += 1
            time.sleep(1 + a)
    return None


# ------------------------------------------------------------------- names
SALTS = (r'\b(?:hydrochloride|hcl|sodium|potassium|calcium|sulfate|sulphate|mesylate|maleate|tartrate|besylate|succinate|'
         r'fumarate|phosphate|acetate|citrate|bromide|nitrate|dihydrate|monohydrate|hydrobromide|magnesium|disodium|axetil)\b')


def canon(name):
    """Canonical lower-case ingredient name: no salts, parentheses or punctuation."""
    s = str(name).lower()
    s = re.sub(r'\(.*?\)', ' ', s)
    s = re.sub(SALTS, ' ', s)
    s = re.sub(r'[^a-z0-9+/\- ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


# ------------------------------------------------------------------- RxNorm
def rx_approx(term, cache, n=5):
    key = f'approx|{term.lower().strip()}'
    if cache.has(key):
        return cache.get(key)
    js = http_get(f'{RXNAV}/approximateTerm.json', {'term': term, 'maxEntries': n})
    if js is None:
        return None
    out = []
    for c in (js.get('approximateGroup') or {}).get('candidate', []) or []:
        out.append({'rxcui': c.get('rxcui'), 'score': float(c.get('score') or 0), 'rank': int(c.get('rank') or 0),
                    'name': c.get('name'), 'source': c.get('source')})
    cache.set(key, out)
    return out


def rx_props(rxcui, cache):
    key = f'props|{rxcui}'
    if cache.has(key):
        return cache.get(key)
    js = http_get(f'{RXNAV}/rxcui/{rxcui}/properties.json')
    if js is None:
        return None
    p = js.get('properties') or {}
    out = {'rxcui': rxcui, 'name': p.get('name'), 'tty': p.get('tty')}
    cache.set(key, out)
    return out


def rx_ingredients(rxcui, cache):
    """Ingredient names of any RxNorm concept (brand, clinical drug, salt form, ingredient)."""
    key = f'ing|{rxcui}'
    if cache.has(key):
        return cache.get(key)
    props = rx_props(rxcui, cache)
    if props is None:
        return None
    if props.get('tty') == 'IN':
        names = [props['name']]
    else:
        js = http_get(f'{RXNAV}/rxcui/{rxcui}/related.json', {'tty': 'IN'})
        if js is None:
            return None
        names = []
        for g in (js.get('relatedGroup') or {}).get('conceptGroup', []) or []:
            if g.get('tty') == 'IN':
                names += [c['name'] for c in g.get('conceptProperties', []) or []]
        if not names and props.get('name'):
            names = [props['name']]
    cache.set(key, names)
    return names


def rx_normalize(text, cache):
    cands = rx_approx(text, cache)
    if not cands:
        return {'query': text, 'rxcui': None, 'score': 0.0, 'matched': None, 'source': None, 'ingredients': []}
    best = max(cands, key=lambda c: (c['score'], -c['rank']))
    ings = rx_ingredients(best['rxcui'], cache) or []
    return {'query': text, 'rxcui': best['rxcui'], 'score': best['score'], 'matched': best['name'],
            'source': best['source'], 'ingredients': sorted({canon(i) for i in ings})}


# ------------------------------------------------- label sentences about cardiac topics
SECTION_GROUP = {   # openFDA label field -> (group, strength)
    'boxed_warning': ('boxed', 3), 'contraindications': ('contraindication', 3),
    'warnings': ('warning', 3), 'warnings_and_cautions': ('warning', 3), 'warnings_and_precautions': ('warning', 3),
    'precautions': ('warning', 3), 'general_precautions': ('warning', 3),
    'adverse_reactions': ('adverse', 2), 'drug_interactions': ('interaction', 2),
    'overdosage': ('overdose', 1), 'clinical_pharmacology': ('pharmacology', 1),
    'pharmacodynamics': ('pharmacology', 1), 'mechanism_of_action': ('pharmacology', 1),
}
TOPICS = {
    'qt': re.compile(r'\bQT(?:c|cF|cB|cI)?\b|[Tt]orsades?'),
    'bradycardia': re.compile(r'[Bb]radycardi|heart rate (?:decreas|reduc|slow)|slow(?:ing|ed|s)? (?:of )?(?:the )?heart rate|sinus (?:pause|arrest)'),
    'av_block': re.compile(r'(?:[Aa]trioventricular|\bA-?V\b) (?:block|conduction)|[Hh]eart block|\bPR (?:interval|prolong)'),
}
_W = r'(?:\w+\s+)'
NEG = [re.compile(p, re.I) for p in (
    rf'\b(?:no|without)\s+{_W}{{0,3}}?(?:effects?|changes?|increases?|prolongation|differences?|impact)\b[^.]{{0,40}}\b(?:QT|torsade|bradycardi|heart block|AV|PR interval)',
    rf'\b(?:did|does|do|was|were|has|have|had|is|are)\s+not\s+{_W}{{0,2}}?(?:prolong\w*|affect\w*|alter\w*|increase\w*|cause\w*|produce\w*|induce\w*|lengthen\w*|associated|shown|show|demonstrate\w*|result\w*|appear\w*|observed|reported|seen|noted)',
    rf'\bnot\s+(?:been\s+)?(?:shown|found|observed|reported|associated)\s+to\s+{_W}{{0,2}}?(?:prolong|affect|alter|increase|cause)',
    rf'\bno\s+(?:evidence|signs?|cases?|reports?)\s+of\s+[^.]{{0,40}}\b(?:QT|torsade|bradycardi|heart block)',
    r'\bQT[^.]{0,60}\b(?:was|were)\s+(?:not|unchanged|similar to placebo)',
)]
POS = re.compile(r'\bprolong\w*|\blengthen\w*|[Tt]orsade|\b(?:increas\w*)\b[^.]{0,60}\bQT|\bQT[^.]{0,60}\b(?:increas\w*|prolong\w*)|'
                 r'\brisk\b|\bcaus\w*|\binduc\w*|\bproduc\w*|\breport\w*|\bobserv\w*|\bsevere|\bsymptomatic|\barrhythmi', re.I)


def split_sentences(text):
    text = re.sub(r'\s+', ' ', text)
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+(?=[A-Z0-9\[(•])', text) if s.strip()]


def classify_sentence(sentence, topic, group):
    """-> 'negated' | 'positive' | 'mention' | None (topic not mentioned)."""
    if not TOPICS[topic].search(sentence):
        return None
    if any(p.search(sentence) for p in NEG):
        return 'negated'
    if group in ('boxed', 'contraindication', 'warning', 'adverse') or POS.search(sentence):
        return 'positive'
    return 'mention'


def extract_label_evidence(label, max_per_topic=15, max_len=420):
    """One openFDA label dict -> {topic: [ {group, strength, field, text, cls} ]} (only sentences about the topic)."""
    out = {t: [] for t in TOPICS}
    for field, (group, strength) in SECTION_GROUP.items():
        v = label.get(field)
        if not v:
            continue
        text = ' '.join(v) if isinstance(v, list) else str(v)
        for s in split_sentences(text):
            for t in TOPICS:
                cls = classify_sentence(s, t, group)
                if cls and len(out[t]) < max_per_topic:
                    out[t].append({'group': group, 'strength': strength, 'field': field, 'cls': cls, 'text': s[:max_len]})
    return out


def label_meta(label):
    o = label.get('openfda') or {}
    gen = o.get('generic_name') or o.get('substance_name') or []
    low = ' '.join(gen).lower()
    single = len(gen) == 1 and not re.search(r'\band\b|,|/|\bwith\b|\+', low)
    return {'set_id': label.get('set_id'), 'id': label.get('id'), 'effective_time': label.get('effective_time'),
            'generic_name': gen[:3], 'brand_name': (o.get('brand_name') or [])[:3], 'single_ingredient': bool(single)}


def fda_fetch(ingredient, cache, api_key=None, limit=10):
    """Retrieve up to `limit` recent labels for an ingredient and cache the extracted evidence. None = network failure."""
    key = f'fda|{ingredient.lower().strip()}|{limit}'
    if cache.has(key):
        return cache.get(key)
    base = {'api_key': api_key} if api_key else {}
    results, failed = None, False
    for field in ('openfda.generic_name', 'openfda.substance_name'):
        params = {**base, 'search': f'{field}:"{ingredient}"', 'limit': limit, 'sort': 'effective_time:desc'}
        js = http_get(FDA_LABEL, params)
        if js and js.get('_status') == 400:           # some deployments reject the sort field
            params.pop('sort')
            js = http_get(FDA_LABEL, params)
        if js is None:
            failed = True; break
        if js.get('results'):
            results = js['results']; break
    if failed:
        return None
    rec = {'ingredient': ingredient, 'labels': []}
    for lab in results or []:
        m = label_meta(lab)
        m['evidence'] = extract_label_evidence(lab)
        rec['labels'].append(m)
    cache.set(key, rec)
    return rec


# ------------------------------------------------------------- evidence tiers
TIER_RANK = {'no_label': -1, 'none': 0, 'negated_only': 1, 'mention': 2, 'weak': 3, 'moderate': 4, 'strong': 5}


def _label_tier(ev):
    pos = [e for e in ev if e['cls'] == 'positive']
    if pos:
        s = max(e['strength'] for e in pos)
        return {3: 'strong', 2: 'moderate', 1: 'weak'}[s]
    if any(e['cls'] == 'mention' for e in ev):
        return 'mention'
    if any(e['cls'] == 'negated' for e in ev):
        return 'negated_only'
    return 'none'


def summarize_evidence(rec, topic, max_labels=5, top_k=3):
    """Aggregate over labels. Prefers single-ingredient labels. 'none' means: no statement found in the retrieved labels
    (NOT proof of safety); 'no_label' means nothing could be retrieved (-> insufficient evidence)."""
    if not rec or not rec.get('labels'):
        return {'topic': topic, 'tier': 'no_label', 'n_labels': 0, 'n_positive': 0, 'combination_only': False, 'evidence': []}
    labs = [l for l in rec['labels'] if l['single_ingredient']]
    combo_only = not labs
    labs = (labs or rec['labels'])[:max_labels]
    tiers = [_label_tier(l['evidence'][topic]) for l in labs]
    tier = max(tiers, key=lambda t: TIER_RANK[t])
    ev = []
    for l, t in zip(labs, tiers):
        for e in l['evidence'][topic]:
            if e['cls'] == 'positive':
                ev.append({**e, 'set_id': l['set_id'], 'effective_time': l['effective_time'], 'brand': l['brand_name'][:1],
                           'url': f"https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid={l['set_id']}" if l['set_id'] else None})
    ev.sort(key=lambda e: -e['strength'])
    return {'topic': topic, 'tier': tier, 'n_labels': len(labs), 'combination_only': combo_only,
            'n_positive': sum(t in ('strong', 'moderate', 'weak') for t in tiers), 'evidence': ev[:top_k]}


# -------------------------------------------------------------------- DDInter
LEVEL_RANK = {'major': 3, 'moderate': 2, 'minor': 1, 'unknown': 0}


def _find(cols, pats):
    for p in pats:
        for c in cols:
            if re.fullmatch(p, c.lower().strip()):
                return c
    return None


class DDI:
    """Pairwise interaction index from DDInter CSVs (any number of files; column names are auto-detected)."""

    def __init__(self, folder):
        self.pairs, self.files, self.columns = {}, [], {}
        paths = sorted(glob.glob(os.path.join(folder, '*.csv'))) if folder and os.path.isdir(folder) else []
        for f in paths:
            df = pd.read_csv(f)
            cols = list(df.columns)
            a = _find(cols, [r'drug_?a(?:_name)?', r'drug[ _]?1', r'drug_?a_?name'])
            b = _find(cols, [r'drug_?b(?:_name)?', r'drug[ _]?2', r'drug_?b_?name'])
            lv = _find(cols, [r'level', r'severity', r'interaction_?level'])
            if not (a and b):
                name_like = [c for c in cols if 'drug' in c.lower() and 'id' not in c.lower()]
                if len(name_like) >= 2:
                    a, b = name_like[:2]
            self.columns[os.path.basename(f)] = {'drug_a': a, 'drug_b': b, 'level': lv, 'all': cols[:12]}
            if not (a and b):
                continue
            self.files.append(os.path.basename(f))
            for x, y, l in zip(df[a], df[b], df[lv] if lv else ['unknown'] * len(df)):
                k = frozenset((canon(x), canon(y)))
                l = str(l).strip().lower()
                l = l if l in LEVEL_RANK else 'unknown'
                if k not in self.pairs or LEVEL_RANK[l] > LEVEL_RANK[self.pairs[k]]:
                    self.pairs[k] = l
        self.drugs = sorted({d for k in self.pairs for d in k})

    def __len__(self): return len(self.pairs)

    def level(self, a, b):
        return self.pairs.get(frozenset((canon(a), canon(b))))

    def check(self, ingredients):
        """All interacting pairs among a list of ingredient names, strongest first."""
        ings = sorted({canon(i) for i in ingredients})
        out = []
        for i in range(len(ings)):
            for j in range(i + 1, len(ings)):
                l = self.pairs.get(frozenset((ings[i], ings[j])))
                if l:
                    out.append({'drug_a': ings[i], 'drug_b': ings[j], 'level': l, 'source': 'DDInter'})
        return sorted(out, key=lambda r: -LEVEL_RANK[r['level']])


# ---------------------------------------------------------- CredibleMeds (optional)
def cm_category(text):
    t = str(text).lower()
    if 'known' in t or t.strip() == 'kr': return 'known'
    if 'possible' in t or t.strip() == 'pr': return 'possible'
    if 'conditional' in t or t.strip() == 'cr': return 'conditional'
    if 'congenital' in t or 'avoid' in t or 'clqt' in t: return 'congenital_avoid'
    return 'other'


def load_crediblemeds(folder):
    """User-supplied CredibleMeds export (csv/xlsx). Returns ({canon_name: category}, info). Used locally only."""
    paths = []
    if folder and os.path.isdir(folder):
        for ext in ('*.csv', '*.xlsx', '*.xls'):
            paths += glob.glob(os.path.join(folder, ext))
    info, mapping = {'files': [os.path.basename(p) for p in paths]}, {}
    for p in paths:
        df = pd.read_csv(p) if p.endswith('.csv') else pd.read_excel(p)
        cols = list(df.columns)
        nc = _find(cols, [r'.*generic.*', r'.*drug.*name.*', r'.*name.*', r'.*drug.*'])
        cc = _find(cols, [r'.*categor.*', r'.*risk.*', r'.*group.*'])
        info[os.path.basename(p)] = {'name_col': nc, 'category_col': cc, 'columns': cols[:10]}
        if nc and cc:
            for n, c in zip(df[nc], df[cc]):
                if isinstance(n, str):
                    mapping[canon(n)] = cm_category(c)
    return mapping, info


In [ ]:
import importlib, json, datetime, numpy as np, pandas as pd
from tqdm.auto import tqdm
import cardiosafe_kb as K
importlib.reload(K)

API_KEY = os.environ.get('OPENFDA_API_KEY')
try:
    from google.colab import userdata
    API_KEY = userdata.get('OPENFDA_API_KEY') or API_KEY
except Exception:
    pass
print('openFDA API key:', 'found' if API_KEY else 'not set (limit: 1,000 requests/day; fine for this notebook)')
rx_cache = K.JsonCache(KB + '/cache_rxnorm.json')
fda_cache = K.JsonCache(KB + '/cache_openfda.json')

## 1. RxNorm: name normalisation
Free-text drug names mapped to RxNorm and then to ingredient names. The last rows are deliberately hard: Indian brand names and a typo. **Look at what RxNorm does with them**, because that tells us whether we need our own brand-to-generic table later.

In [ ]:
tests = ['azithromycin', 'Zithromax', 'Zofran', 'ondansetron 4 mg', 'ondansetrone', 'metoprolol succinate', 'Augmentin 625',
         'Azithral', 'Dolo 650', 'Ecosprin']
rows = []
for t in tests:
    r = K.rx_normalize(t, rx_cache)
    rows.append({'input': t, 'matched': r['matched'], 'score': r['score'], 'ingredients': ', '.join(r['ingredients'])})
res = pd.DataFrame(rows); display(res)
rx_cache.save()
if res['matched'].isna().all():
    print('RxNorm returned nothing for any query: the API may be unreachable from this runtime. Send me the error and I will adapt.')

## 2. openFDA label evidence: sanity check
For each drug the notebook retrieves up to 10 recent FDA labels, prefers single-ingredient ones, extracts every sentence about the three cardiac topics, and assigns an evidence **tier**:

- **strong:** an affirmative statement in a boxed warning, contraindication or warnings section
- **moderate:** in adverse reactions or drug interactions
- **weak:** only in pharmacology or overdose text
- **mention / negated_only / none:** QT is mentioned without a risk statement / only "no effect" statements / no statement found
- **no_label:** nothing could be retrieved (this must lead to "insufficient evidence", never to "safe")

**'none' means "no statement found in the retrieved labels", not "proven safe".**

The expected-positive and expected-negative drugs below are my own small list of well-known cases. It is a **sanity check, not a gold standard**: please read the printed evidence and tell me about any drug where you disagree.

In [ ]:
POS = {'qt': ['azithromycin', 'clarithromycin', 'erythromycin', 'levofloxacin', 'moxifloxacin', 'ciprofloxacin', 'ondansetron', 'haloperidol',
              'citalopram', 'escitalopram', 'methadone', 'amiodarone', 'sotalol', 'hydroxychloroquine', 'fluconazole', 'quetiapine', 'donepezil', 'chlorpromazine'],
       'bradycardia': ['metoprolol', 'atenolol', 'propranolol', 'digoxin', 'verapamil', 'diltiazem', 'amiodarone', 'donepezil'],
       'av_block': ['metoprolol', 'atenolol', 'propranolol', 'digoxin', 'verapamil', 'diltiazem', 'amiodarone']}
NEG = ['metformin', 'atorvastatin', 'lisinopril', 'amlodipine', 'levothyroxine', 'acetaminophen', 'simvastatin', 'gabapentin']
SANITY = sorted(set(sum(POS.values(), []) + NEG))
FLAGGED = ('strong', 'moderate', 'weak')

records, rows = {}, []
for d in tqdm(SANITY):
    rec = K.fda_fetch(d, fda_cache, API_KEY)
    records[d] = rec
    row = {'drug': d, 'n_labels': len(rec['labels']) if rec else None, 'single_ingredient_labels': sum(l['single_ingredient'] for l in rec['labels']) if rec else None}
    for t in K.TOPICS:
        row[t] = K.summarize_evidence(rec, t)['tier'] if rec else 'FETCH_FAILED'
    rows.append(row)
fda_cache.save()
tiers = pd.DataFrame(rows).set_index('drug'); display(tiers)
print('API requests this session:', K.STATS)

In [ ]:
summ = []
for t in K.TOPICS:
    pos = [d for d in POS[t] if tiers.loc[d, t] != 'FETCH_FAILED']; neg = [d for d in NEG if tiers.loc[d, t] != 'FETCH_FAILED']
    hit = [d for d in pos if tiers.loc[d, t] in FLAGGED]; fp = [d for d in neg if tiers.loc[d, t] in FLAGGED]
    summ.append({'topic': t, 'expected_positive': len(pos), 'flagged': len(hit), 'detection_rate': len(hit) / max(len(pos), 1),
                 'missed': ', '.join(sorted(set(pos) - set(hit))), 'expected_negative': len(neg), 'false_flags': len(fp), 'false_flag_drugs': ', '.join(fp)})
display(pd.DataFrame(summ).set_index('topic').round(2))

In [ ]:
# READ THESE: the actual evidence behind each QT flag (strongest sentence per drug)
for d in POS['qt'] + NEG:
    s = K.summarize_evidence(records[d], 'qt')
    print(f"\n{d.upper():18s} tier={s['tier']:12s} labels={s['n_labels']} (positive in {s['n_positive']})" + ('  [combination labels only]' if s['combination_only'] else ''))
    for e in s['evidence'][:1]:
        print(f"   [{e['group']}] {e['text'][:230]}")

In [ ]:
# Same for the two cardiac-conduction topics, for a few drugs, to check the rules generalise
for d in ['metoprolol', 'digoxin', 'verapamil', 'donepezil', 'metformin']:
    for t in ('bradycardia', 'av_block'):
        s = K.summarize_evidence(records[d], t)
        e = s['evidence'][0]['text'][:200] if s['evidence'] else '-'
        print(f'{d:11s} {t:11s} {s["tier"]:12s} {e}')

## 3. DDInter interactions
Loads whatever CSV files you put in `kb/ddinter/`, auto-detects the drug-name and severity columns and prints what it detected so we can verify it.

In [ ]:
ddi = K.DDI(KB + '/ddinter')
if len(ddi) == 0:
    print('No DDInter data loaded. Put the downloaded CSV files in', KB + '/ddinter', 'and re-run this cell.')
    print('Detected columns per file:', ddi.columns)
else:
    print('files:', ddi.files)
    for f, c in ddi.columns.items(): print(' ', f, '->', {k: v for k, v in c.items() if k != 'all'})
    print(f'{len(ddi):,} unique drug pairs | {len(ddi.drugs):,} distinct drug names')
    display(pd.Series(list(ddi.pairs.values())).value_counts().to_frame('pairs'))
    inn = [d for d in SANITY if K.canon(d) in set(ddi.drugs)]
    print(f'sanity drugs present in DDInter: {len(inn)}/{len(SANITY)} | missing: {sorted(set(SANITY) - set(inn))}')
    qtpairs = ddi.check(POS['qt'])
    print(f'\nInteractions DDInter lists among the {len(POS["qt"])} QT-label drugs: {len(qtpairs)} pairs')
    display(pd.DataFrame(qtpairs).head(15))

## 4. Optional: agreement with CredibleMeds
Only runs if you placed your own CredibleMeds export in `kb/crediblemeds/`. We ask: of the drugs experts classify as Known / Possible / Conditional QT risk, how many does our label-based extractor flag, and how often does it flag drugs that are not on the list? Only aggregate numbers are saved; the list itself is never copied or redistributed.

In [ ]:
cm, cm_info = K.load_crediblemeds(KB + '/crediblemeds')
if not cm:
    print('CredibleMeds list not provided (or columns not detected). Detected:', cm_info)
else:
    print('detected columns:', {k: v for k, v in cm_info.items() if k != 'files'})
    print('category counts:', pd.Series(list(cm.values())).value_counts().to_dict())
    rng = np.random.default_rng(0)
    risky = [d for d, c in cm.items() if c in ('known', 'possible', 'conditional')]
    risky = [d for d in risky if ' ' not in d and '+' not in d]
    sample_pos = list(rng.choice(risky, min(120, len(risky)), replace=False))
    pool_neg = [d for d in (ddi.drugs if len(ddi) else NEG) if d not in cm and ' ' not in d and '+' not in d]
    sample_neg = list(rng.choice(pool_neg, min(60, len(pool_neg)), replace=False))
    print(f'fetching {len(sample_pos)} listed + {len(sample_neg)} unlisted drugs (about {2 * (len(sample_pos) + len(sample_neg))} API calls at most)')
    out = []
    for d in tqdm(sample_pos + sample_neg):
        rec = K.fda_fetch(d, fda_cache, API_KEY)
        if rec is None: continue
        out.append({'drug': d, 'cm': cm.get(d, 'not_listed'), 'tier': K.summarize_evidence(rec, 'qt')['tier']})
    fda_cache.save()
    r = pd.DataFrame(out); r['flagged'] = r['tier'].isin(FLAGGED)
    tab = r.groupby('cm').agg(n=('drug', 'count'), flagged=('flagged', 'sum'), no_label=('tier', lambda s: int((s == 'no_label').sum())))
    tab['flagged_rate'] = tab['flagged'] / tab['n']
    display(tab.round(3)); tab.to_csv(KB + '/results_07_crediblemeds_agreement.csv')
    print('Good behaviour: high flagged_rate for known (and possible), lower for conditional, low for not_listed. Misses on known-risk drugs are the cases to inspect.')
    print('Known-risk drugs we did NOT flag:', r[(r.cm == 'known') & ~r.flagged].drug.tolist())

## 5. Export for the next notebooks and the demo

In [ ]:
rx_cache.save(); fda_cache.save()
kb = {'retrieved': datetime.date.today().isoformat(), 'drugs': {}}
for d, rec in records.items():
    if rec is None: continue
    kb['drugs'][d] = {t: K.summarize_evidence(rec, t) for t in K.TOPICS}
json.dump(kb, open(KB + '/kb_sanity.json', 'w'), indent=1)
json.dump({'retrieved': kb['retrieved'], 'openfda_labels_per_drug': 10, 'max_labels_used': 5, 'flag_tiers': list(FLAGGED),
           'ddinter_files': ddi.files, 'ddinter_pairs': len(ddi), 'crediblemeds_provided': bool(cm)},
          open(KB + '/kb_config.json', 'w'), indent=1)
print('saved kb_sanity.json, kb_config.json, caches in', KB)

## Next
Send me, as text:
1. The RxNorm table (this tells us what to do about brand names).
2. The two summary tables (tiers per drug, detection summary).
3. The printed QT evidence block (so we can inspect real label sentences and fix the rules).
4. The DDInter detection printout (files, detected columns, counts).
5. The CredibleMeds agreement table, if you provided the list.

Then notebook 08: the prescription pipeline (synthetic prescription images, OCR comparison, drug extraction), evaluated against known ground truth.